# VQ-VAE-2 Latents — Content vs Style Disentanglement

Content/style channel splits, spatial activation maps, cross-view alignment, and patch retrieval.

In [ ]:
import sys
import os

# Add project root to sys.path (this notebook lives in eval/)
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import json

# Import local modules
import models.vqvae as vqvae
import utils.utils as utils
from utils.utils import load_data, CreateBrainMaskd, ApplyBrainMaskd

# ============================================================
# CONFIGURATION
# ============================================================
# Use vqvae_best.pt for the best checkpoint (by rolling avg total loss),
# or vqvae_model.pt for the latest checkpoint.
CHECKPOINT_PATH = (
    "/home/ng24/projects/multiview-crl/results/ADNI_registered/multiview-06-content-lr-002-single-level/vqvae_best1.pt"
)
DATA_DIR = "/data/natalia/ADNI_stripped"
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
NUM_SAMPLES = 200  # number of subjects to process
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ADNI 2-view split (must match training)
CONTENT_SIZE = 256  # dimensions treated as content (shared between T1/T2)
STYLE_SIZE = 256  # dimensions treated as style (view-specific)

# Resampling: "original" (1 mm, ~182x218x182) or "2mm" (91x109x91)
RESAMPLE_MODE = "original"
# ============================================================

print(f"Using device: {DEVICE}")

## 1. Load checkpoint & build model

In [ ]:
# Load settings saved alongside the checkpoint
settings_path = os.path.join(os.path.dirname(CHECKPOINT_PATH), "settings.json")
with open(settings_path, "r") as f:
    settings = json.load(f)

print("Model settings:")
for k in [
    "vqvae_hidden_channels",
    "vqvae_res_channels",
    "vqvae_nb_res_layers",
    "vqvae_nb_levels",
    "vqvae_embed_dim",
    "vqvae_nb_entries",
    "vqvae_scaling_rates",
]:
    print(f"  {k}: {settings.get(k, 'N/A')}")

# Override CONTENT_SIZE / STYLE_SIZE from saved settings.
# settings.json stores content_indices (list[list[int]]) and style_indices (list[int])
# written by update_args() at training time — use them so the VQVAE is rebuilt
# with exactly the same content/style ratio that was used during training.
# Falling back to the hardcoded values only when the keys are absent (old checkpoints).
if "content_indices" in settings and "style_indices" in settings:
    CONTENT_SIZE = len(settings["content_indices"][0])
    STYLE_SIZE = len(settings["style_indices"])
    print(f"\n  content_size : {CONTENT_SIZE}  (from settings.json['content_indices'])")
    print(f"  style_size   : {STYLE_SIZE}  (from settings.json['style_indices'])")
else:
    print(f"\n  ⚠ content_indices / style_indices not found in settings.json.")
    print(f"    Using hardcoded CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}.")
    print(f"    If these don't match the training run, content_channels will be wrong.")

In [ ]:
# Build model — must match the exact architecture used during training.
# inject_style_to_decoder controls whether decoders[0] has a final_conv branch;
# if it differs from the checkpoint the decoder weights will silently not load.
inject_style = settings.get("inject_style_to_decoder", False)

# ── Auto-detect content/style split from checkpoint weights ──────────────────
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["encoders"]

# Strip MoCo "online." prefix if present to find the right keys
_prefix = ""
if any(k.startswith("online.") for k in state_dict):
    _prefix = "online."

# channel_logits is now a ParameterDict keyed by level.
# Old checkpoints: "module.channel_logits" (single param)
# New checkpoints: "module.channel_logits.0", "module.channel_logits.1", etc.
_cl_key_old = f"{_prefix}module.channel_logits"

# Find all channel_logits.X keys where X is a digit
_cl_levels = []
for k in state_dict:
    stripped = k[len(_prefix) :] if k.startswith(_prefix) else k
    if stripped.startswith("module.channel_logits."):
        suffix = stripped[len("module.channel_logits.") :]
        if suffix.isdigit():
            _cl_levels.append(int(suffix))
_cl_levels = sorted(_cl_levels)

# ── Detect mask_mode from checkpoint keys ────────────────────────────────────
# fixed mode: has fixed_mask_* buffers, no channel_logits
# learned mode: has channel_logits params
# onthefly mode: neither (logits computed from activations at runtime)
_has_fixed_mask = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.fixed_mask_") for k in state_dict
)

if _has_fixed_mask:
    _detected_mask_mode = "fixed"
elif _cl_levels or _cl_key_old in state_dict:
    _detected_mask_mode = "learned"
else:
    _detected_mask_mode = "onthefly"

# Allow settings.json to override (it's always authoritative if present)
_mask_mode = settings.get("mask_mode", _detected_mask_mode)

if _cl_levels:
    # New-style per-level channel_logits
    _cl_key_0 = f"{_prefix}module.channel_logits.{_cl_levels[0]}"
    _mask_dim = state_dict[_cl_key_0].shape[0]
    content_style_levels = _cl_levels
elif _cl_key_old in state_dict:
    # Old-style single channel_logits
    _mask_dim = state_dict[_cl_key_old].shape[0]
    content_style_levels = [0]
else:
    _mask_dim = None
    # No channel_logits (onthefly or fixed mode) — read content_style_levels from settings
    content_style_levels = settings.get("content_style_levels", [0])

# ── Per-level content_channels detection from codebook input sizes ────────────
hidden_channels = settings["vqvae_hidden_channels"]
_embed_dim = settings["vqvae_embed_dim"]
_nb_levels = settings["vqvae_nb_levels"]

# Detect per-level content_channels from codebook conv_in dimensions.
# Works for any mask_mode (learned, onthefly, or fixed) as long as the codebook was
# sized for the content subset (content_channels < hidden_channels).
_content_ch_per_level = {}
_used_codebook_detection = False
content_ratios = None

for lvl in content_style_levels:
    _cb_key = f"{_prefix}module.codebooks.{lvl}.conv_in.weight"
    if _cb_key in state_dict:
        _cb_in = state_dict[_cb_key].shape[1]
        if lvl == _nb_levels - 1:
            # Top level: codebook input = content_channels (no embed_dim concat)
            _content_ch_per_level[lvl] = _cb_in
        else:
            # Lower levels: codebook input = content_channels + embed_dim
            _content_ch_per_level[lvl] = _cb_in - _embed_dim

# Check if codebook detection is valid: if ALL detected levels give
# content_channels == hidden_channels, the checkpoint predates per-level
# codebook sizing — fall back to settings-based CONTENT_SIZE / STYLE_SIZE.
_all_full_width = all(cc == hidden_channels for cc in _content_ch_per_level.values()) if _content_ch_per_level else True

if not _all_full_width:
    # Codebook was sized for content masking — use detected values
    _used_codebook_detection = True

    _first_lvl = content_style_levels[0]
    _content_channels = _content_ch_per_level.get(_first_lvl, hidden_channels)
    _style_channels = hidden_channels - _content_channels

    CONTENT_SIZE = _content_channels
    STYLE_SIZE = _style_channels

    # Compute per-level content_ratios if levels have different content sizes
    _ratios = [_content_ch_per_level[lvl] / hidden_channels for lvl in content_style_levels]
    if len(set(_ratios)) > 1:
        content_ratios = _ratios

    _mode_desc = {
        "learned": "learned",
        "onthefly": "onthefly (no channel_logits)",
        "fixed": "fixed (first K = content)",
    }
    print(f"Auto-detected from checkpoint (per-level codebook sizing):")
    print(f"  mask_mode             : {_mode_desc.get(_mask_mode, _mask_mode)}")
    print(f"  content_style_levels  : {content_style_levels}")
    for lvl in content_style_levels:
        cc = _content_ch_per_level.get(lvl, "?")
        print(f"  level {lvl} content_ch    : {cc} / {hidden_channels}  ({cc/hidden_channels:.1%})")
    print(f"  -> CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}")
    if content_ratios:
        print(f"  -> content_ratios={content_ratios}")
else:
    # Old checkpoint: codebooks use full hidden_channels.
    # Keep CONTENT_SIZE / STYLE_SIZE from settings cell (cell above).
    print(f"Codebook detection gives content_ch == hidden_channels at all levels.")
    print(f"  content_style_levels  : {content_style_levels}")
    print(f"  → Using CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE} from settings.json")

# ── Detect whether checkpoint uses hidden_channels or embed_dim masking ──────
if _mask_dim is not None and _mask_dim != hidden_channels:
    raise RuntimeError(
        f"Checkpoint has channel_logits of size {_mask_dim} but hidden_channels={hidden_channels}. "
        f"This checkpoint was likely trained with an older code version where the mask was on "
        f"embed_dim={_embed_dim}. You need to checkout the matching code version "
        f"to load this checkpoint."
    )

# ── Detect separate_encoders from checkpoint keys ────────────────────────────
# If the checkpoint contains "module.encoders_v1.*" keys, the model was trained
# with --separate-encoders.
_sep_enc = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.encoders_v1.") for k in state_dict
)
if _sep_enc:
    print("Detected separate_encoders in checkpoint (encoders_v1 keys found).")

_style_inj_mode = settings.get("style_injection_mode", "concat")

# ── Read training flags that affect the encoder forward pass ─────────────────
# These MUST match training exactly, otherwise the encoder produces different
# features at inference time (e.g. style channels zeroed between levels when
# they should be passed through, causing out-of-distribution inputs at higher levels).
_pass_full = settings.get("pass_full_to_next_level", False)
_narrow_enc = settings.get("narrow_encoder_input", False)
_top_recon = settings.get("top_level_recon_only", False)
_quantize_style = settings.get("quantize_style", False)
_style_embed_dim = settings.get("style_embed_dim", None)
_style_nb_entries_raw = settings.get("style_nb_entries", None)
_use_content_proj = settings.get("use_content_projection", False)

# ── Normalize per-level codebook sizes ───────────────────────────────────────
# --vqvae-nb-entries / --style-nb-entries are argparse nargs="+" → always a
# list in settings.json. W&B sweeps occasionally round-trip these as
# stringified lists ("[384]") or nested lists ([[384]]), which then reach
# CodeLayer where torch.randn(embed_dim, nb_entries) raises
#   TypeError: randn(): argument 'size' failed to unpack ... got list
# Flatten + coerce to int / flat list[int] before handing to VQVAE.
import ast as _ast


def _normalize_entries(v):
    if v is None:
        return None
    if isinstance(v, str):
        try:
            v = _ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return int(v)
    if isinstance(v, (list, tuple)):
        flat = []
        for x in v:
            if isinstance(x, (list, tuple)):
                flat.extend(x)
            else:
                flat.append(x)
        flat = [int(x) for x in flat]
        return flat[0] if len(flat) == 1 else flat
    return int(v)


_nb_entries = _normalize_entries(settings["vqvae_nb_entries"])
_style_nb_entries = _normalize_entries(_style_nb_entries_raw)
print(f"vqvae_nb_entries (normalized) : {_nb_entries}")
if _style_nb_entries is not None:
    print(f"style_nb_entries (normalized) : {_style_nb_entries}")

# ── Optional: skip levels in the final reconstruction decoder ────────────────
# Define SKIP_RECON_LEVELS in a cell above to ablate finer codes, e.g.
#     SKIP_RECON_LEVELS = [0, 1]
# zeros out levels 0 and 1 in the final (level-0) decoder concat, leaving only
# the top (coarsest) codes to drive reconstruction. The top level itself cannot
# be skipped. If undefined, falls back to whatever was stored at training time.
try:
    _skip_recon = SKIP_RECON_LEVELS  # noqa: F821 — optional user override
    print(f"⚠ Notebook override: skip_decoder_concat_levels={_skip_recon}")
except NameError:
    _skip_recon = settings.get("skip_decoder_concat_levels", None)
    if _skip_recon:
        print(f"skip_decoder_concat_levels (from settings): {_skip_recon}")

vqvae_model = vqvae.VQVAE(
    in_channels=1,
    hidden_channels=hidden_channels,
    res_channels=settings["vqvae_res_channels"],
    nb_res_layers=settings.get("vqvae_nb_res_layers", 2),
    nb_levels=_nb_levels,
    embed_dim=_embed_dim,
    nb_entries=_nb_entries,
    scaling_rates=settings["vqvae_scaling_rates"],
    content_size=CONTENT_SIZE,
    style_size=STYLE_SIZE,
    inject_style_to_decoder=inject_style,
    content_style_levels=content_style_levels,
    content_ratios=content_ratios,
    separate_encoders=_sep_enc,
    mask_mode=_mask_mode,
    style_injection_mode=_style_inj_mode,
    pass_full_to_next_level=_pass_full,
    narrow_encoder_input=_narrow_enc,
    top_level_recon_only=_top_recon,
    quantize_style=_quantize_style,
    style_embed_dim=_style_embed_dim,
    style_nb_entries=_style_nb_entries,
    use_content_projection=_use_content_proj,
    skip_decoder_concat_levels=_skip_recon,
)

content_channels = vqvae_model.content_channels
print(f"\nhidden_channels         : {hidden_channels}")
print(f"content_channels        : {content_channels}")
if vqvae_model.content_channels_per_level:
    print(f"content_channels_per_lvl: {vqvae_model.content_channels_per_level}")
print(f"content_style_levels    : {content_style_levels}")
print(f"mask_mode               : {_mask_mode}")
print(f"inject_style_to_decoder : {inject_style}")
print(f"separate_encoders       : {_sep_enc}")
print(f"style_injection_mode    : {_style_inj_mode}")
print(f"pass_full_to_next_level : {_pass_full}")
print(f"narrow_encoder_input    : {_narrow_enc}")
print(f"top_level_recon_only    : {_top_recon}")
print(f"quantize_style          : {_quantize_style}")
print(f"use_content_projection  : {_use_content_proj}")

# Wrap in DataParallel to match the saved checkpoint structure
vqvae_model = torch.nn.DataParallel(vqvae_model, device_ids=[0])

# ── Key remapping ─────────────────────────────────────────────────────────────
new_state_dict = {}
is_moco_checkpoint = any(k.startswith("online.") for k in state_dict)
if is_moco_checkpoint:
    print("Detected MoCoEncoder checkpoint — stripping 'online.' prefix.")

for k, v in state_dict.items():
    k = k.replace(".blocks.", ".stack.")  # legacy rename
    # Migrate old single channel_logits → ParameterDict key "0"
    if k.endswith("module.channel_logits") and not any(c.isdigit() for c in k.split(".")[-1]):
        k = k + ".0"
    if k.startswith("online."):
        new_state_dict[k[len("online.") :]] = v
    elif k.startswith(
        ("momentum_encoders.", "momentum_codebook_projs.", "momentum_content_proj.", "momentum_level0_proj.", "queue_")
    ):
        pass  # MoCo-only state — discard
    elif k.startswith("momentum_encoders_v1."):
        pass  # MoCo momentum copy of view-1 encoder — discard
    else:
        new_state_dict[k] = v

# ── Drop keys with shape mismatches (e.g. projection head from older config) ─
_model_sd = vqvae_model.state_dict()
_shape_skipped = []
for k in list(new_state_dict):
    if k in _model_sd and new_state_dict[k].shape != _model_sd[k].shape:
        _shape_skipped.append(f"{k}: ckpt {new_state_dict[k].shape} vs model {_model_sd[k].shape}")
        del new_state_dict[k]
if _shape_skipped:
    print(f"\u26a0 Skipped {len(_shape_skipped)} keys with shape mismatch:")
    for s in _shape_skipped:
        print(f"    {s}")

missing, unexpected = vqvae_model.load_state_dict(new_state_dict, strict=False)

missing_real = [k for k in missing if "num_batches_tracked" not in k]
unexpected_real = [k for k in unexpected if "num_batches_tracked" not in k]

if missing_real:
    print(f"\u26a0 Missing keys ({len(missing_real)}): {missing_real[:6]}{'...' if len(missing_real) > 6 else ''}")
if unexpected_real:
    print(
        f"\u26a0 Unexpected keys ({len(unexpected_real)}): {unexpected_real[:6]}{'...' if len(unexpected_real) > 6 else ''}"
    )
if not missing_real and not unexpected_real:
    print("\u2713 Checkpoint loaded cleanly")
elif not missing_real:
    print("\u2713 Checkpoint loaded (unexpected keys are harmless extras in the file)")

vqvae_model.to(DEVICE)
vqvae_model.eval()
print(f"\u2713 Model on {DEVICE}, step {checkpoint['step']}")
print(f"  Total params: {sum(p.numel() for p in vqvae_model.parameters()):,}")

In [ ]:
# -- Diagnostic: isolate where the constant-output problem lives --
import torch

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# 1. Check encoder weight statistics (are weights non-trivial?)
print("=== Encoder weight check ===")
for name, p in list(vqvae_module.encoders[0].named_parameters())[:6]:
    print(
        f"  {name}: shape={tuple(p.shape)}  "
        f"mean={p.data.mean():.6f}  std={p.data.std():.6f}  "
        f"absmax={p.data.abs().max():.6f}"
    )

# 2. Check ReZero alpha values (if 0, residual blocks are identity)
print("\n=== ReZero alpha values ===")
for name, p in vqvae_module.encoders[0].named_parameters():
    if "alpha" in name:
        print(f"  {name}: {p.data.item():.6f}")

# 3. Test raw encoder (bypass VQVAE forward)
print("\n=== Raw encoder test ===")
with torch.no_grad():
    d1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    d2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)

    # Run through encoder level 0 directly
    out1 = vqvae_module.encoders[0](d1)
    out2 = vqvae_module.encoders[0](d2)

    print(f"  Encoder output shape: {out1.shape}")
    print(f"  out1 range: [{out1.min():.4f}, {out1.max():.4f}]  mean={out1.mean():.6f}")
    print(f"  out2 range: [{out2.min():.4f}, {out2.max():.4f}]  mean={out2.mean():.6f}")

    # Check spatial variation (different voxels should have different values)
    print(f"  out1 spatial std (per-channel): {out1[0].std(dim=[1, 2, 3]).mean():.6f}")

    # Pool and compare
    p1 = out1.mean(dim=[2, 3, 4])
    p2 = out2.mean(dim=[2, 3, 4])
    cos = torch.nn.functional.cosine_similarity(p1, p2).item()
    print(f"  Pooled cosine similarity: {cos:.6f}")
    if cos > 0.999:
        print("  WARNING: Raw encoder also produces constant output!")
    else:
        print("  OK: Raw encoder works - bug is in VQVAE.forward()")

# 4. Test layer by layer to find where signal dies
print("\n=== Layer-by-layer signal propagation ===")
with torch.no_grad():
    x1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    x2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    for layer_idx, layer in enumerate(vqvae_module.encoders[0].layers):
        x1 = layer(x1)
        x2 = layer(x2)
        cos = torch.nn.functional.cosine_similarity(x1.flatten(1), x2.flatten(1)).item()
        print(
            f"  After layer {layer_idx} ({type(layer).__name__}): "
            f"shape={tuple(x1.shape)}  cos={cos:.6f}  "
            f"range=[{x1.min():.3f},{x1.max():.3f}]"
        )

## 2. Load dataset & transforms

In [ ]:
df = pd.read_csv(CSV_PATH)
label_values = sorted(df["Group"].unique())
label_map = {v: i for i, v in enumerate(label_values)}
label_names = {i: v for v, i in label_map.items()}
print(f"Labels: {label_map}")

items, missing = load_data(df, DATA_DIR, label_map)
print(f"Loaded {len(items)} samples, {len(missing)} missing files")
items = items[:NUM_SAMPLES]
print(f"Using first {len(items)} subjects")

In [ ]:
# Match the training val pipeline EXACTLY. Previously the notebook always
# applied CreateBrainMaskd (intensity > 0 threshold). Training with masks on
# disk skips that and loads the proper brain-mask .nii.gz directly via
# LoadImaged — a different mask → different NormalizeIntensityd stats →
# different features → different modality probe result. Fix: delegate to
# utils.utils.transforms and use masks_from_disk when the CSV items carry
# mask paths.
from utils.utils import transforms as get_transforms

# Read spacing and spatial_size from settings.json (saved at training time)
spacing = settings.get("image_spacing", 2.0)
crop_margin = settings.get("crop_margin", 0)

_saved_spatial = settings.get("spatial_size", None)
if _saved_spatial is not None:
    spatial_size = tuple(_saved_spatial)
elif spacing == 1.0:
    spatial_size = (182, 218, 182)
else:
    spatial_size = (91, 109, 91)

# Mirror data/datasets.py:1283 — masks_from_disk is True iff every loaded
# item carries both mask paths.
masks_from_disk = all(("mask_image" in it) and ("mask_z_image" in it) for it in items)

_, _val_transforms_raw = get_transforms(
    spacing=spacing,
    crop_margin=crop_margin,
    spatial_size=spatial_size,
    masks_from_disk=masks_from_disk,
    asymmetric_aug=False,
)

# The raw val transform expects:
#   - ``mask_t1``/``mask_t2`` paths when ``masks_from_disk=True`` (loaded by LoadImaged)
#   - a ``label`` key at the end (required by the final ToTensord in utils.utils.transforms)
# Existing notebook cells pass only ``image_t1``/``image_t2``. Wrap the
# transform to auto-inject mask paths and a placeholder label, keyed on the
# T1 image path.
_img_to_extras = {
    it["image"]: {
        "mask_t1": it.get("mask_image"),
        "mask_t2": it.get("mask_z_image"),
        "label": it.get("label", 0),
    }
    for it in items
}


def val_transforms(data_dict):
    d = dict(data_dict)
    extras = _img_to_extras.get(d.get("image_t1"), {})
    if masks_from_disk:
        if "mask_t1" not in d and extras.get("mask_t1") is not None:
            d["mask_t1"] = extras["mask_t1"]
        if "mask_t2" not in d and extras.get("mask_t2") is not None:
            d["mask_t2"] = extras["mask_t2"]
    if "label" not in d:
        d["label"] = extras.get("label", 0)
    return _val_transforms_raw(d)


print(f"Transforms ready — spacing={spacing}mm, spatial_size={spatial_size}, " f"masks_from_disk={masks_from_disk}")
if not masks_from_disk:
    print(
        "  (No disk masks found on items — using intensity-threshold CreateBrainMaskd.\n"
        "  If training ran with disk masks, the modality probe will differ from the training log.)"
    )

## 3. Feature extraction

We call `vqvae_model(img, pool_only=True, return_recon=False)` which returns:
- `encoder_features`: list of `(B, embed_dim)` pooled vectors, one per level
  - All levels are pooled from the codebook projection (embed_dim space)
  - When `channel_logits` is active, content/style split is in this embed_dim space
- `estimated_content_indices`: the Gumbel-selected embedding dim indices at level 0

For visualisation we split level-0 pooled features into **content** and **style** subsets
using `estimated_content_indices`.

In [ ]:
nb_levels = settings["vqvae_nb_levels"]

# Storage
all_features = {f"level_{i}": [] for i in range(nb_levels)}
all_labels = []
all_modalities = []
all_subjects = []
all_content_indices = {}  # dict mapping level -> view-0 (T1) content indices
all_content_indices_v1 = {}  # dict mapping level -> view-1 (T2) content indices (only set for per-view masks)

print(f"Extracting latent features from {len(items)} subjects (T1 + T2 each) ...")

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# Map modality name → view_idx for separate-encoder models.
# view 0 = T1 (first modality), view 1 = T2 (second modality).
_VIEW_IDX = {"T1": 0, "T2": 1}

# Detect per-view masks: separate encoders with channel_logits_v1
_has_per_view = (
    getattr(vqvae_module, "separate_encoders", False) and getattr(vqvae_module, "channel_logits_v1", None) is not None
)

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            # Get soft_content_masks (7th returned element)
            _, _, enc_features, _, _, _, soft_masks, *_ = vqvae_model(
                img,
                return_recon=False,
                pool_only=True,
                view_idx=_VIEW_IDX.get(modality, 0),
            )

        # enc_features is a list of (1, hidden_channels) pooled tensors per level.
        for level_idx, pooled in enumerate(enc_features):
            all_features[f"level_{level_idx}"].append(pooled.squeeze(0).cpu().float().numpy())

        # Record the content indices per level.
        #  - Tuple masks: training-style n_views=2 with per-view masks (unused
        #    here since we run n_views=1, but handled for completeness).
        #  - Single-tensor mask: single-view forward.  When per-view learned
        #    masks are active the returned tensor is the view-specific mask
        #    (channel_logits_v1 when view_idx=1), so T2 must be stored in
        #    all_content_indices_v1 to preserve the per-view split.
        for lvl, mask in soft_masks.items():
            if isinstance(mask, tuple):
                mask_v0, mask_v1 = mask
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask_v0.bool())[-1].tolist()
                elif modality == "T2" and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask_v1.bool())[-1].tolist()
            else:
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask.bool())[-1].tolist()
                elif modality == "T2" and _has_per_view and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask.bool())[-1].tolist()

        all_labels.append(item["label"])
        all_modalities.append(modality)
        all_subjects.append(item.get("subject", idx))

# Convert to numpy
for key in all_features:
    all_features[key] = np.array(all_features[key])
all_labels = np.array(all_labels)
all_modalities = np.array(all_modalities)

print(f"\nDone. {len(all_labels)} samples total.")
for key, val in all_features.items():
    print(f"  {key}: {val.shape}")

# Derive content / style subsets from the hidden_channels features for each masked level.
all_style_indices = {}
all_style_indices_v1 = {}

for lvl in all_content_indices.keys():
    all_style_indices[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices[lvl])]
    if _has_per_view and lvl in all_content_indices_v1:
        all_style_indices_v1[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices_v1[lvl])]
    else:
        all_style_indices_v1[lvl] = all_style_indices[lvl]

if len(all_content_indices.get(0, [])) > 0:
    for lvl in all_content_indices.keys():
        print(f"\n--- Level {lvl} ---")
        print(f"  content indices v0 ({len(all_content_indices[lvl])} ch): {all_content_indices[lvl][:8]}...")
        print(f"  style indices   v0 ({len(all_style_indices[lvl])} ch):   {all_style_indices[lvl][:8]}...")
        if _has_per_view and lvl in all_content_indices_v1:
            print(f"  content indices v1 ({len(all_content_indices_v1[lvl])} ch): {all_content_indices_v1[lvl][:8]}...")
            print(f"  style indices   v1 ({len(all_style_indices_v1[lvl])} ch):   {all_style_indices_v1[lvl][:8]}...")

## 7. Content vs Style channel visualisation (level 0)

The Gumbel mask at level 0 selects `content_channels` out of `embed_dim` dimensions.
If the model has learned to disentangle content (shared across T1/T2) from style
(view-specific), the **content** PCA should show T1 and T2 mixed together, while
the **style** PCA should separate them clearly.

In [ ]:
if 0 in all_content_indices and len(all_style_indices.get(0, [])) > 0:
    level0_feats = all_features["level_0"]
    content_feats = level0_feats[:, all_content_indices[0]]
    style_feats = level0_feats[:, all_style_indices[0]]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for row, (feats, title_sfx) in enumerate(
        [
            (content_feats, f"Content ({len(all_content_indices.get(0, []))} ch)"),
            (style_feats, f"Style   ({len(all_style_indices.get(0, []))} ch)"),
        ]
    ):
        pca = PCA(n_components=2)
        f2d = pca.fit_transform(feats)
        ev = pca.explained_variance_ratio_

        # Col 0 — by diagnosis
        ax = axes[row, 0]
        for lab_idx, lab_name in label_names.items():
            m = all_labels == lab_idx
            ax.scatter(f2d[m, 0], f2d[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.65, s=18)
        ax.set_title(f"{title_sfx} — Diagnosis\n({ev.sum()*100:.1f}% var)")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

        # Col 1 — by modality
        ax = axes[row, 1]
        for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
            m = all_modalities == mod
            ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
        ax.set_title(f"{title_sfx} — Modality")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    plt.suptitle("Level 0 — Content vs Style (encoder hidden_channels)", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig("content_vs_style_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Saved content_vs_style_pca.png")
else:
    print("Skipped — channel_logits not active or all channels are content.")

In [ ]:
# ── Spatial code maps: content vs style at level 0 ───────────────────────────
N_EXAMPLES = 3  # number of subjects to visualise
has_style_cb = len(vqvae_module.style_codebooks) > 0

print(f"Style codebooks active: {has_style_cb}")
if 0 in all_content_indices:
    print(f"Content channels (level 0): {len(all_content_indices.get(0, []))}/{hidden_channels}")
    print(f"Style channels   (level 0): {len(all_style_indices.get(0, []))}/{hidden_channels}")

# Pick example subjects (one per diagnosis class if possible)
example_indices = []
seen_labels = set()
for i, item in enumerate(items):
    if item["label"] not in seen_labels:
        example_indices.append(i)
        seen_labels.add(item["label"])
    if len(example_indices) >= N_EXAMPLES:
        break
# Fill remaining slots if fewer classes than N_EXAMPLES
for i in range(len(items)):
    if len(example_indices) >= N_EXAMPLES:
        break
    if i not in example_indices:
        example_indices.append(i)

print(f"\nExtracting spatial code maps for {len(example_indices)} subjects ...")

for ex_idx in example_indices:
    item = items[ex_idx]
    subj = item.get("subject", ex_idx)
    label = label_names[item["label"]]

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            recon, diffs, _, _, _, id_outputs_raw, _, style_id_out = vqvae_model(
                img,
                return_recon=True,
                pool_only=False,
                view_idx=0,
            )

        # id_outputs_raw is coarsest-first; reverse to finest-first
        id_outputs_ex = id_outputs_raw[::-1]
        content_codes = id_outputs_ex[0].squeeze(0).cpu().numpy()  # (D, H, W)

        style_codes = None
        if has_style_cb and 0 in style_id_out:
            style_codes = style_id_out[0].squeeze(0).cpu().numpy()

        # Determine number of columns: 3 slices x (content + style if available)
        n_maps = 2 if style_codes is not None else 1
        fig, axes = plt.subplots(n_maps, 3, figsize=(15, 5 * n_maps), squeeze=False)
        fig.suptitle(
            f"Subject {subj} ({label}, {modality}) — Level 0 Spatial Code Maps\n"
            f"Code map shape: {content_codes.shape}",
            fontsize=13,
            fontweight="bold",
        )

        # Slice indices (middle of each axis)
        mid_d, mid_h, mid_w = [s // 2 for s in content_codes.shape]
        slices_content = [
            ("Axial (d={})".format(mid_d), content_codes[mid_d, :, :]),
            ("Coronal (h={})".format(mid_h), content_codes[:, mid_h, :]),
            ("Sagittal (w={})".format(mid_w), content_codes[:, :, mid_w]),
        ]

        for col, (slice_name, slice_data) in enumerate(slices_content):
            ax = axes[0, col]
            im = ax.imshow(slice_data, cmap="tab20", interpolation="nearest", aspect="auto")
            ax.set_title(f"Content — {slice_name}")
            ax.set_xlabel("dim 1")
            ax.set_ylabel("dim 0")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        if style_codes is not None:
            slices_style = [
                ("Axial (d={})".format(mid_d), style_codes[mid_d, :, :]),
                ("Coronal (h={})".format(mid_h), style_codes[:, mid_h, :]),
                ("Sagittal (w={})".format(mid_w), style_codes[:, :, mid_w]),
            ]
            for col, (slice_name, slice_data) in enumerate(slices_style):
                ax = axes[1, col]
                im = ax.imshow(slice_data, cmap="tab20", interpolation="nearest", aspect="auto")
                ax.set_title(f"Style — {slice_name}")
                ax.set_xlabel("dim 1")
                ax.set_ylabel("dim 0")
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plt.tight_layout()
        plt.show()

        # ── Per-entry frequency comparison: content vs style codebook ────────
        if style_codes is not None:
            fig, axes = plt.subplots(1, 2, figsize=(16, 4))
            fig.suptitle(f"Subject {subj} ({label}) — Codebook Entry Frequency (Level 0)", fontsize=12)

            content_nb = vqvae_module.codebooks[0].n_embed
            style_nb = vqvae_module.style_codebooks["0"].n_embed

            c_hist = np.bincount(content_codes.ravel(), minlength=content_nb).astype(float)
            c_hist /= c_hist.sum()
            axes[0].bar(range(content_nb), c_hist, color="steelblue", alpha=0.7)
            axes[0].set_title(f"Content codebook ({content_nb} entries, {len(all_content_indices.get(0, []))} ch)")
            axes[0].set_xlabel("Entry index")
            axes[0].set_ylabel("Frequency")

            s_hist = np.bincount(style_codes.ravel(), minlength=style_nb).astype(float)
            s_hist /= s_hist.sum()
            axes[1].bar(range(style_nb), s_hist, color="tomato", alpha=0.7)
            axes[1].set_title(f"Style codebook ({style_nb} entries, {len(all_style_indices.get(0, []))} ch)")
            axes[1].set_xlabel("Entry index")
            axes[1].set_ylabel("Frequency")

            plt.tight_layout()
            plt.show()

            used_c = (c_hist > 0).sum()
            used_s = (s_hist > 0).sum()
            print(f"  {subj}: content uses {used_c}/{content_nb} entries, " f"style uses {used_s}/{style_nb} entries")
        else:
            # No style codebook — just show content entry histogram
            content_nb = vqvae_module.codebooks[0].n_embed
            c_hist = np.bincount(content_codes.ravel(), minlength=content_nb).astype(float)
            c_hist /= c_hist.sum()
            used_c = (c_hist > 0).sum()

            fig, ax = plt.subplots(figsize=(10, 3))
            ax.bar(range(content_nb), c_hist, color="steelblue", alpha=0.7)
            ax.set_title(
                f"Subject {subj} ({label}) — Content Codebook Entry Frequency "
                f"({used_c}/{content_nb} used, {len(all_content_indices.get(0, []))} content ch)"
            )
            ax.set_xlabel("Entry index")
            ax.set_ylabel("Frequency")
            plt.tight_layout()
            plt.show()

    del recon, diffs, id_outputs_raw, style_id_out

print("\n✓ Spatial code map visualisation complete.")

## 7a. Spatial Activation Maps — Content vs Style Channels (Level 0)

For a few example subjects, extract the **full spatial encoder output** at level 0 and split it
into content and style channels. Visualise the mean activation (and top individual channels)
across axial, coronal, and sagittal mid-slices to see what spatial patterns each subspace captures.

In [ ]:
# ── Spatial activation maps: content vs style channels at level 0 ────────────
N_EXAMPLES = 3  # one per diagnosis class

# Pick one subject per class
example_indices = []
seen_labels = set()
for i, item in enumerate(items):
    if item["label"] not in seen_labels:
        example_indices.append(i)
        seen_labels.add(item["label"])
    if len(example_indices) >= N_EXAMPLES:
        break

content_idx = np.array(all_content_indices[0]) if 0 in all_content_indices else np.arange(hidden_channels)
style_idx = np.array(all_style_indices.get(0, [])) if len(all_style_indices.get(0, [])) > 0 else None

print(f"Content channels: {len(content_idx)}, Style channels: {len(style_idx) if style_idx is not None else 0}")
print(f"Visualising {len(example_indices)} subjects ...\n")

# Run encoder only (no codebook/decoder) to get spatial feature maps
enc_stack = vqvae_module.encoders
if vqvae_module.separate_encoders and vqvae_module.encoders_v1 is not None:
    enc_stack_v1 = vqvae_module.encoders_v1
else:
    enc_stack_v1 = enc_stack

TOP_K_CHANNELS = 4  # number of individual channels to show

for ex_idx in example_indices:
    item = items[ex_idx]
    subj = item.get("subject", ex_idx)
    label = label_names[item["label"]]

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)
    img = transformed["image_t1"].unsqueeze(0).to(DEVICE)

    # Forward through encoder stack, collect output at each level
    enc_input = img
    spatial_outputs = []
    with torch.no_grad():
        for enc in enc_stack:
            enc_input = enc(enc_input)
            spatial_outputs.append(enc_input)

    # Level 0 spatial features: (1, hidden_channels, D, H, W)
    feat_map = spatial_outputs[0].squeeze(0).cpu().numpy()  # (C, D, H, W)

    content_map = feat_map[content_idx]  # (n_content, D, H, W)
    style_map = feat_map[style_idx] if style_idx is not None and len(style_idx) > 0 else None

    # Mid-slice indices
    _, D, H, W = feat_map.shape
    mid_d, mid_h, mid_w = D // 2, H // 2, W // 2

    # ── Mean activation maps ────────────────────────────────────────────────
    n_rows = 2 if style_map is not None else 1
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows), squeeze=False)
    fig.suptitle(
        f"Subject {subj} ({label}) — Level 0 Mean Activation Maps\n"
        f"Feature map: {feat_map.shape[0]} ch × {D}×{H}×{W}",
        fontsize=13,
        fontweight="bold",
    )

    mean_content = content_map.mean(axis=0)  # (D, H, W)
    for col, (name, slc) in enumerate(
        [
            (f"Axial (d={mid_d})", mean_content[mid_d]),
            (f"Coronal (h={mid_h})", mean_content[:, mid_h]),
            (f"Sagittal (w={mid_w})", mean_content[:, :, mid_w]),
        ]
    ):
        ax = axes[0, col]
        im = ax.imshow(slc, cmap="inferno", interpolation="bilinear", aspect="auto")
        ax.set_title(f"Content mean ({len(content_idx)} ch) — {name}")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    if style_map is not None:
        mean_style = style_map.mean(axis=0)
        for col, (name, slc) in enumerate(
            [
                (f"Axial (d={mid_d})", mean_style[mid_d]),
                (f"Coronal (h={mid_h})", mean_style[:, mid_h]),
                (f"Sagittal (w={mid_w})", mean_style[:, :, mid_w]),
            ]
        ):
            ax = axes[1, col]
            im = ax.imshow(slc, cmap="inferno", interpolation="bilinear", aspect="auto")
            ax.set_title(f"Style mean ({len(style_idx)} ch) — {name}")
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    # ── Top individual channels by variance (most informative) ──────────────
    for subset_name, subset_map, idx_arr in [
        ("Content", content_map, content_idx),
        ("Style", style_map, style_idx),
    ]:
        if subset_map is None:
            continue
        # Rank channels by spatial variance
        ch_var = subset_map.reshape(subset_map.shape[0], -1).var(axis=1)
        top_ch = np.argsort(ch_var)[::-1][:TOP_K_CHANNELS]

        fig, axes = plt.subplots(TOP_K_CHANNELS, 3, figsize=(14, 3.5 * TOP_K_CHANNELS), squeeze=False)
        fig.suptitle(
            f"Subject {subj} ({label}) — Top-{TOP_K_CHANNELS} {subset_name} Channels by Variance (Level 0)",
            fontsize=13,
            fontweight="bold",
        )

        for row, ch_local in enumerate(top_ch):
            ch_global = idx_arr[ch_local]
            ch_data = subset_map[ch_local]  # (D, H, W)
            for col, (name, slc) in enumerate(
                [
                    (f"Axial (d={mid_d})", ch_data[mid_d]),
                    (f"Coronal (h={mid_h})", ch_data[:, mid_h]),
                    (f"Sagittal (w={mid_w})", ch_data[:, :, mid_w]),
                ]
            ):
                ax = axes[row, col]
                im = ax.imshow(slc, cmap="inferno", interpolation="bilinear", aspect="auto")
                if col == 0:
                    ax.set_ylabel(f"Ch {ch_global}\n(var={ch_var[ch_local]:.4f})", fontsize=9)
                if row == 0:
                    ax.set_title(name)
                plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        plt.tight_layout()
        plt.show()

    del spatial_outputs, feat_map

print("✓ Spatial activation map visualisation complete.")

## 7a-ii. Content vs Style Overlay — Colocalization Map

Overlay content and style activation maps on the same MRI slice so we can directly compare
*where* each subspace is active. Uses **L2-norm across channels** as the saliency per voxel
(standard activation-map reduction), upsampled to input resolution and shown as:

- **Red channel** = content activation (normalized per-map)
- **Blue channel** = style activation (normalized per-map)
- **Purple** = both active, **pure red/blue** = one stream dominates

Shown per-subject (one per class) and as a **batch-mean** saliency across all extracted
subjects to smooth out per-subject noise.


In [ ]:
# ── Content vs Style overlay: per-subject + batch-mean saliency ───────────────
import torch.nn.functional as F

content_idx_t = torch.as_tensor(all_content_indices[0], dtype=torch.long) if 0 in all_content_indices else None
style_idx_t = (
    torch.as_tensor(all_style_indices.get(0, []), dtype=torch.long) if len(all_style_indices.get(0, [])) > 0 else None
)

assert (
    content_idx_t is not None and style_idx_t is not None and len(style_idx_t) > 0
), "Need both content and style channels at level 0 for overlay."

enc_stack = vqvae_module.encoders


def _spatial_saliency(feat_map, idx):
    """L2 norm across the given channel subset -> (D, H, W) saliency."""
    sub = feat_map.index_select(0, idx.to(feat_map.device))  # (n, D, H, W)
    return sub.pow(2).sum(dim=0).sqrt()  # (D, H, W)


def _upsample_to(vol, target_shape):
    """vol: (D,H,W) tensor -> (D',H',W') nearest-to-target via trilinear."""
    v = vol[None, None].float()
    up = F.interpolate(v, size=target_shape, mode="trilinear", align_corners=False)
    return up[0, 0]


def _norm01(x, eps=1e-8):
    x = x - x.min()
    m = x.max()
    return x / (m + eps)


def _rgb_overlay(bg, content_sal, style_sal, alpha=0.55):
    """bg, content_sal, style_sal: 2D arrays same shape. Returns (H,W,3) RGB image."""
    bg = _norm01(bg)
    c = _norm01(content_sal)
    s = _norm01(style_sal)
    base = np.stack([bg, bg, bg], axis=-1)
    overlay = np.stack([c, np.zeros_like(c), s], axis=-1)  # R=content, B=style
    return (1 - alpha) * base + alpha * overlay


# Pick one subject per class (reuse logic from previous cell)
example_indices = []
seen_labels = set()
for i, item in enumerate(items):
    if item["label"] not in seen_labels:
        example_indices.append(i)
        seen_labels.add(item["label"])
    if len(example_indices) >= 3:
        break

# Accumulators for batch-mean (across ALL items, both modalities)
sum_content_sal = None
sum_style_sal = None
n_acc = 0
per_subject_cache = {}  # ex_idx -> (img_np, content_sal_np, style_sal_np)

print(f"Computing saliency maps across {len(items)} samples ...")
for i, item in enumerate(items):
    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)
    img = transformed["image_t1"].unsqueeze(0).to(DEVICE)

    enc_input = img
    with torch.no_grad():
        for enc in enc_stack:
            enc_input = enc(enc_input)
            break  # only need level 0 output
        feat = enc_input.squeeze(0)  # (C, D, H, W)

        content_sal = _spatial_saliency(feat, content_idx_t)
        style_sal = _spatial_saliency(feat, style_idx_t)

        # Upsample to input resolution
        target = img.shape[-3:]
        content_sal_up = _upsample_to(content_sal, target).cpu().numpy()
        style_sal_up = _upsample_to(style_sal, target).cpu().numpy()

    # Per-sample max-normalize before accumulating so subjects with larger
    # activation magnitudes do not dominate the batch mean (each subject contributes
    # equally to the "where does the model typically look" spatial pattern).
    content_sal_n = content_sal_up / (content_sal_up.max() + 1e-8)
    style_sal_n = style_sal_up / (style_sal_up.max() + 1e-8)

    if sum_content_sal is None:
        sum_content_sal = np.zeros_like(content_sal_n)
        sum_style_sal = np.zeros_like(style_sal_n)
    sum_content_sal += content_sal_n
    sum_style_sal += style_sal_n
    n_acc += 1

    if i in example_indices:
        per_subject_cache[i] = (img.squeeze().cpu().numpy(), content_sal_up, style_sal_up)

mean_content_sal = sum_content_sal / n_acc
mean_style_sal = sum_style_sal / n_acc
print(f"Accumulated {n_acc} samples.\n")


# ── Plot per-subject overlays + batch mean ───────────────────────────────────
def _plot_overlay_row(axes_row, img_np, c_sal, s_sal, title_prefix):
    D, H, W = img_np.shape
    md, mh, mw = D // 2, H // 2, W // 2
    views = [
        ("Axial", img_np[md], c_sal[md], s_sal[md]),
        ("Coronal", img_np[:, mh], c_sal[:, mh], s_sal[:, mh]),
        ("Sagittal", img_np[:, :, mw], c_sal[:, :, mw], s_sal[:, :, mw]),
    ]
    for col, (name, bg, c, s) in enumerate(views):
        ax = axes_row[col]
        ax.imshow(_rgb_overlay(bg, c, s), interpolation="bilinear", aspect="auto")
        ax.set_title(f"{title_prefix} — {name}")
        ax.set_xticks([])
        ax.set_yticks([])


n_rows = len(per_subject_cache) + 1  # +1 for batch mean (uses subject-0 bg)
fig, axes = plt.subplots(n_rows, 3, figsize=(14, 4.2 * n_rows), squeeze=False)
fig.suptitle(
    "Content (red) vs Style (blue) Activation Overlay — Level 0\n"
    "Purple = both active, red = content-only, blue = style-only",
    fontsize=13,
    fontweight="bold",
)

for row, ex_idx in enumerate(per_subject_cache):
    img_np, c_sal, s_sal = per_subject_cache[ex_idx]
    label = label_names[items[ex_idx]["label"]]
    subj = items[ex_idx].get("subject", ex_idx)
    _plot_overlay_row(axes[row], img_np, c_sal, s_sal, f"{subj} ({label})")

# Batch mean row — use the first cached subject's MRI as background for anatomy reference
bg_img = next(iter(per_subject_cache.values()))[0]
_plot_overlay_row(axes[-1], bg_img, mean_content_sal, mean_style_sal, f"Batch mean (n={n_acc})")

# Legend
fig.text(
    0.5,
    -0.01,
    "Red channel = content L2-norm saliency  |  Blue channel = style L2-norm saliency  "
    "(per-sample max-normalized before averaging; overlay re-normalized to [0,1] per map)",
    ha="center",
    fontsize=9,
    style="italic",
)

plt.tight_layout()
plt.show()


# ── Side-by-side: pure content vs pure style for the batch mean ──────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8), squeeze=False)
fig.suptitle(f"Batch-Mean Saliency — Content vs Style (n={n_acc})", fontsize=13, fontweight="bold")

D, H, W = mean_content_sal.shape
md, mh, mw = D // 2, H // 2, W // 2
slices = [("Axial", md, 0), ("Coronal", mh, 1), ("Sagittal", mw, 2)]

for col, (name, mid, axis) in enumerate(slices):
    c_slice = np.take(mean_content_sal, mid, axis=axis)
    s_slice = np.take(mean_style_sal, mid, axis=axis)
    im0 = axes[0, col].imshow(c_slice, cmap="Reds", interpolation="bilinear", aspect="auto")
    axes[0, col].set_title(f"Content mean — {name}")
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046, pad=0.04)
    im1 = axes[1, col].imshow(s_slice, cmap="Blues", interpolation="bilinear", aspect="auto")
    axes[1, col].set_title(f"Style mean — {name}")
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("✓ Overlay + batch-mean saliency visualisation complete.")

## 7b. Per-level modality separation: content vs style

Quantitative check: for each level, split features into content/style channels using
the Gumbel mask, then measure how strongly modality (T1/T2) clusters in each subspace.

- **Silhouette score** close to 1 → strong modality clustering (bad for content, good for style)
- **Silhouette score** close to 0 → modalities are mixed (good for content)

If the contrastive loss is working at a given level, the content silhouette should be
lower than the style silhouette at that level.

In [ ]:
from sklearn.metrics import silhouette_score

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# Gather per-level content/style indices from the model's channel_logits or fixed masks
modality_binary = (all_modalities == "T2").astype(int)

# Check for per-view masks
_has_v1_logits = hasattr(vqvae_module, "channel_logits_v1") and vqvae_module.channel_logits_v1 is not None
_is_fixed = getattr(vqvae_module, "mask_mode", "onthefly") == "fixed"

results = []

for lvl in range(nb_levels):
    feats = all_features[f"level_{lvl}"]
    n_ch = feats.shape[1]

    # Determine content/style indices for this level.
    # Three paths: fixed mask, learned/onthefly with channel_logits, or no mask.
    _has_logits_for_lvl = (
        hasattr(vqvae_module, "channel_logits")
        and len(vqvae_module.channel_logits) > 0
        and str(lvl) in vqvae_module.channel_logits
    )
    _has_fixed_for_lvl = _is_fixed and hasattr(vqvae_module, f"fixed_mask_{lvl}")

    if _has_fixed_for_lvl:
        # Fixed mode: first k channels = content, rest = style
        k = vqvae_module.content_channels_per_level.get(lvl, vqvae_module.content_channels)
        c_idx_v0 = np.arange(k)
        s_idx_v0 = np.arange(k, n_ch)
        c_idx_v1 = c_idx_v0  # fixed mode is always shared across views
        s_idx_v1 = s_idx_v0

        content_feats = feats[:, c_idx_v0]
        style_feats_v0 = feats[:, s_idx_v0] if len(s_idx_v0) > 0 else None
        style_feats_v1 = None

        sil_content = silhouette_score(content_feats, modality_binary) if content_feats.shape[1] >= 2 else float("nan")
        if style_feats_v0 is not None:
            sil_style = (
                silhouette_score(style_feats_v0, modality_binary) if style_feats_v0.shape[1] >= 2 else float("nan")
            )
        else:
            sil_style = float("nan")
        sil_all = silhouette_score(feats, modality_binary)

        results.append(
            {
                "level": lvl,
                "content_ch": k,
                "style_ch": n_ch - k,
                "sil_content": sil_content,
                "sil_style": sil_style,
                "sil_all": sil_all,
            }
        )
        print(
            f"Level {lvl} (fixed):  content sil={sil_content:+.3f} ({k} ch)  |  "
            f"style sil={sil_style:+.3f} ({n_ch - k} ch)  |  "
            f"all sil={sil_all:+.3f}"
        )

    elif _has_logits_for_lvl:
        # Learned / onthefly: use channel_logits to find top-k
        logits_v0 = vqvae_module.channel_logits[str(lvl)].detach().cpu()
        k = vqvae_module.content_channels_per_level.get(lvl, vqvae_module.content_channels)
        c_idx_v0 = logits_v0.topk(k).indices.sort().values.numpy()
        s_idx_v0 = np.array([i for i in range(n_ch) if i not in set(c_idx_v0)])

        if _has_v1_logits and str(lvl) in vqvae_module.channel_logits_v1:
            logits_v1 = vqvae_module.channel_logits_v1[str(lvl)].detach().cpu()
            c_idx_v1 = logits_v1.topk(k).indices.sort().values.numpy()
            s_idx_v1 = np.array([i for i in range(n_ch) if i not in set(c_idx_v1)])

            # Per-view: select correct content/style for each modality
            t1_m = all_modalities == "T1"
            t2_m = all_modalities == "T2"
            content_feats = np.empty((len(feats), k))
            content_feats[t1_m] = feats[t1_m][:, c_idx_v0]
            content_feats[t2_m] = feats[t2_m][:, c_idx_v1]
            # Style dims may differ in count; use all dims for 'all' silhouette
            style_feats_v0 = feats[t1_m][:, s_idx_v0] if len(s_idx_v0) > 0 else None
            style_feats_v1 = feats[t2_m][:, s_idx_v1] if len(s_idx_v1) > 0 else None
        else:
            # Shared mask
            content_feats = feats[:, c_idx_v0]
            style_feats_v0 = feats[:, s_idx_v0] if len(s_idx_v0) > 0 else None
            style_feats_v1 = None
            c_idx_v1 = c_idx_v0
            s_idx_v1 = s_idx_v0

        sil_content = silhouette_score(content_feats, modality_binary) if content_feats.shape[1] >= 2 else float("nan")

        # For style silhouette with per-view masks, use v0 indices for all
        # (style dims may differ; this is an approximation)
        if style_feats_v0 is not None and not _has_v1_logits:
            style_feats_all = feats[:, s_idx_v0]
            sil_style = (
                silhouette_score(style_feats_all, modality_binary) if style_feats_all.shape[1] >= 2 else float("nan")
            )
        else:
            sil_style = float("nan")  # per-view style dims differ, skip

        sil_all = silhouette_score(feats, modality_binary)

        results.append(
            {
                "level": lvl,
                "content_ch": k,
                "style_ch": n_ch - k,
                "sil_content": sil_content,
                "sil_style": sil_style,
                "sil_all": sil_all,
            }
        )
        per_view_note = " (per-view)" if _has_v1_logits else ""
        print(
            f"Level {lvl}{per_view_note}:  content sil={sil_content:+.3f} ({k} ch)  |  "
            f"style sil={sil_style:+.3f} ({n_ch - k} ch)  |  "
            f"all sil={sil_all:+.3f}"
        )
    else:
        sil_all = silhouette_score(feats, modality_binary)
        results.append(
            {
                "level": lvl,
                "content_ch": n_ch,
                "style_ch": 0,
                "sil_content": sil_all,
                "sil_style": float("nan"),
                "sil_all": sil_all,
            }
        )
        print(f"Level {lvl}:  no mask  |  all sil={sil_all:+.3f} ({n_ch} ch)")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results))
w = 0.25
bars_c = ax.bar(x - w, [r["sil_content"] for r in results], w, label="Content", color="steelblue")
bars_s = ax.bar(x, [r["sil_style"] for r in results], w, label="Style", color="tomato")
bars_a = ax.bar(x + w, [r["sil_all"] for r in results], w, label="All", color="gray", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f"Level {r['level']}" for r in results])
ax.set_ylabel("Silhouette score (modality)")
ax.set_title("Modality separation by content/style subspace per level\n(lower content = better disentanglement)")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.legend()
plt.tight_layout()
plt.savefig("modality_silhouette_per_level.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved modality_silhouette_per_level.png")

## 7b. Cross-View Content Alignment (Identifiability Test)

**Core test:** If content is identifiable, the content representation of a T1 scan
should match the content representation of the paired T2 scan of the **same brain**.

For each subject we compute:
- **Content cosine similarity** between T1 and T2 (same subject) → should be **high**
- **Style cosine similarity** between T1 and T2 (same subject) → should be **low**
  (style encodes modality-specific contrast, which differs between T1 and T2)

We also compare against **cross-subject** baselines (random T1–T2 pairings) to show
that the high content similarity is subject-specific, not a degenerate constant mapping.

Additionally, we report **Recall@k** for cross-modality retrieval: given a T1 content
vector, can we retrieve the correct T2 from a gallery by nearest-neighbor search?

In [ ]:
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# ── Split features into paired T1/T2 per subject ────────────────
# Data alternates: T1_subj0, T2_subj0, T1_subj1, T2_subj1, ...
level0_feats = all_features["level_0"]
n_subjects = len(level0_feats) // 2
t1_idx = np.arange(0, 2 * n_subjects, 2)
t2_idx = np.arange(1, 2 * n_subjects, 2)

assert np.all(all_modalities[t1_idx] == "T1"), "T1/T2 alternation assumption broken"
assert np.all(all_modalities[t2_idx] == "T2"), "T1/T2 alternation assumption broken"

content_idx = list(all_content_indices[0]) if 0 in all_content_indices else []
content_idx_v1 = list(all_content_indices_v1[0]) if 0 in all_content_indices_v1 else content_idx
style_idx = all_style_indices.get(0, [])
style_idx_v1 = all_style_indices_v1.get(0, []) if _has_per_view else style_idx

print(f"Subjects: {n_subjects}")
print(f"Content dims v0: {len(content_idx)},  Style dims v0: {len(style_idx)}")
if _has_per_view:
    print(f"Content dims v1: {len(content_idx_v1)},  Style dims v1: {len(style_idx_v1)}")
    overlap = set(content_idx) & set(content_idx_v1)
    print(f"Content channel overlap: {len(overlap)} / {len(content_idx)}  (indices shared by both views)")

# ── Extract content and style features ───────────────────────────
# With per-view masks, v0 and v1 select DIFFERENT physical channels as "content".
# Comparing raw indexed channels across views is meaningless — the k-th content
# dimension of v0 has no correspondence to the k-th content dimension of v1.
#
# Instead, we compare in the ALIGNED k-dimensional space: each view's features
# are masked and then the selected channels are extracted, producing k-dim
# vectors that the contrastive loss actually operates on.
t1_content = level0_feats[t1_idx][:, content_idx]  # (N, k)
t2_content = level0_feats[t2_idx][:, content_idx_v1]  # (N, k)
t1_style = level0_feats[t1_idx][:, style_idx]  # (N, style_dim)
t2_style = level0_feats[t2_idx][:, style_idx_v1]

# Per-level content/style indices and paired features
level_content = {}
for lvl in range(nb_levels):
    feats = all_features[f"level_{lvl}"]
    c_idx = list(all_content_indices[lvl]) if lvl in all_content_indices else []
    c_idx_v1 = list(all_content_indices_v1[lvl]) if lvl in all_content_indices_v1 else c_idx
    s_idx = all_style_indices.get(lvl, [])
    s_idx_v1 = all_style_indices_v1.get(lvl, s_idx) if _has_per_view else s_idx
    level_content[lvl] = {
        "t1": feats[t1_idx],
        "t2": feats[t2_idx],
        "content_idx": c_idx,
        "content_idx_v1": c_idx_v1,
        "style_idx": s_idx,
        "style_idx_v1": s_idx_v1,
    }


# ── 1. Same-subject cosine similarity ───────────────────────────
def paired_cosine_sim(A, B):
    """Row-wise cosine similarity between paired rows of A and B."""
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-8)
    return np.sum(A_norm * B_norm, axis=1)


same_content_sim = paired_cosine_sim(t1_content, t2_content)
same_style_sim = paired_cosine_sim(t1_style, t2_style)

# Per-level same-subject similarity (all dims at that level)
same_level_sim = {}
for lvl in range(nb_levels):
    same_level_sim[lvl] = paired_cosine_sim(level_content[lvl]["t1"], level_content[lvl]["t2"])

# ── 2. Cross-subject baseline ───────────────────────────────────
# For per-view masks: cross-subject similarity must also use
# per-view indexed features (t1 with content_idx, t2 with content_idx_v1).
content_sim_matrix = sklearn_cosine(t1_content, t2_content)  # (N, N)
style_sim_matrix = sklearn_cosine(t1_style, t2_style)


# ── 3. L2-normalised comparison (what the MoCo loss actually sees) ──
# The contrastive loss L2-normalises content features before computing
# dot products. Replicate that here for a fair comparison.
def l2_normalize(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.clip(norms, 1e-8, None)


t1_content_norm = l2_normalize(t1_content)
t2_content_norm = l2_normalize(t2_content)

same_content_sim_norm = np.sum(t1_content_norm * t2_content_norm, axis=1)
cross_content_sim_norm = sklearn_cosine(t1_content_norm, t2_content_norm)

# ── 4. Print summary ────────────────────────────────────────────
print(f"\nSame-subject cosine similarity (mean +/- std):")
print(f"  Content: {same_content_sim.mean():.3f} +/- {same_content_sim.std():.3f}")
print(f"  Style:   {same_style_sim.mean():.3f} +/- {same_style_sim.std():.3f}")

print(f"\nSame-subject cosine similarity by level (all dims):")
for lvl in sorted(same_level_sim.keys()):
    s = same_level_sim[lvl]
    print(f"  Level {lvl}: {s.mean():.3f} +/- {s.std():.3f}")

# Cross-subject: mean of off-diagonal entries
n = content_sim_matrix.shape[0]
off_diag_mask = ~np.eye(n, dtype=bool)
cross_content_mean = content_sim_matrix[off_diag_mask].mean()
cross_content_std = content_sim_matrix[off_diag_mask].std()
cross_style_mean = style_sim_matrix[off_diag_mask].mean()
cross_style_std = style_sim_matrix[off_diag_mask].std()

print(f"\nCross-subject cosine similarity (mean +/- std):")
print(f"  Content: {cross_content_mean:.3f} +/- {cross_content_std:.3f}")
print(f"  Style:   {cross_style_mean:.3f} +/- {cross_style_std:.3f}")

# Diagonal = same-subject in the cross-subject matrix
diag_content = np.diag(content_sim_matrix)
diag_style = np.diag(style_sim_matrix)
print(f"\nSanity check — diagonal of cross-subject matrix (= same-subject):")
print(f"  Content: {diag_content.mean():.3f} +/- {diag_content.std():.3f}")
print(f"  Style:   {diag_style.mean():.3f} +/- {diag_style.std():.3f}")

gap = diag_content.mean() - cross_content_mean
print(
    f"\nContent same-vs-cross gap: {gap:+.3f}  {'(GOOD: same > cross)' if gap > 0.05 else '(WARNING: no meaningful gap)'}"
)

# L2-normalised (what loss sees)
print(f"\nL2-normalised content similarity (matches MoCo loss):")
print(f"  Same-subject:  {same_content_sim_norm.mean():.3f} +/- {same_content_sim_norm.std():.3f}")
cross_norm_off = cross_content_sim_norm[off_diag_mask]
print(f"  Cross-subject: {cross_norm_off.mean():.3f} +/- {cross_norm_off.std():.3f}")
gap_norm = same_content_sim_norm.mean() - cross_norm_off.mean()
print(f"  Gap: {gap_norm:+.3f}")

if _has_per_view:
    print(f"\nNOTE: Per-view masks are active. v0 content channels: {content_idx[:5]}...")
    print(f"  v1 content channels: {content_idx_v1[:5]}...")
    print(f"  These are DIFFERENT physical channels. Cosine similarity is computed")
    print(f"  in the k-dim aligned space (position-wise, not channel-identity-wise).")
    print(f"  If channels have no natural ordering correspondence, this metric")
    print(f"  may underestimate true alignment. Consider the per-level (all dims)")
    print(f"  similarities above as a more reliable signal.")

## 7b-ii. Cross-Modal Patch Retrieval Accuracy (Spatial)

For each spatial patch in modality A (T1), find the nearest-neighbour patch in modality B (T2)
across the **entire batch** using cosine similarity on the **content channels** of the spatial
feature map. If content is properly aligned, the retrieved patch should come from the
**same subject and same spatial position**.

Reports:
- **Position-accurate Recall@k**: retrieved patch is from the correct subject AND spatial position
- **Subject-accurate Recall@k**: retrieved patch is from the correct subject (any position)
- **Chance level** for comparison

In [ ]:
import torch.nn.functional as F

# ── Configuration ────────────────────────────────────────────────────────────
PATCH_GRID = settings.get("patch_grid", [4, 5, 4])  # (D, H, W) — must match training
TOP_K = [1, 5, 10]

# ── Extract spatial feature maps for all subjects ────────────────────────────
enc_stack_v0 = vqvae_module.encoders
enc_stack_v1 = (
    vqvae_module.encoders_v1
    if vqvae_module.separate_encoders and vqvae_module.encoders_v1 is not None
    else enc_stack_v0
)

content_idx = list(all_content_indices[0]) if 0 in all_content_indices else list(range(hidden_channels))
content_idx_v1 = list(all_content_indices_v1[0]) if 0 in all_content_indices_v1 else content_idx

print(f"Patch grid: {PATCH_GRID}  =>  {np.prod(PATCH_GRID)} patches per volume")
print(f"Content channels v0: {len(content_idx)}, v1: {len(content_idx_v1)}")
print(f"Extracting spatial features for {len(items)} subjects ...")

t1_patches_all = []  # will be (n_subjects, n_patches, n_content_ch)
t2_patches_all = []

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)
        encs = enc_stack_v0 if modality == "T1" else enc_stack_v1

        with torch.no_grad():
            x = img
            for enc in encs:
                x = enc(x)
                break  # level 0 only

        # x: (1, C, D', H', W') — spatial feature map at level 0
        feat = x  # keep on device for adaptive_avg_pool3d

        # Pool to patch grid
        gD, gH, gW = PATCH_GRID
        patches = F.adaptive_avg_pool3d(feat, (gD, gH, gW))  # (1, C, gD, gH, gW)
        patches = patches.squeeze(0)  # (C, gD, gH, gW)

        # Select content channels and reshape to (n_patches, n_content)
        ci = content_idx if modality == "T1" else content_idx_v1
        content_patches = patches[ci]  # (n_content, gD, gH, gW)
        content_patches = content_patches.reshape(len(ci), -1).T  # (n_patches, n_content)
        content_patches = content_patches.cpu().float().numpy()

        if modality == "T1":
            t1_patches_all.append(content_patches)
        else:
            t2_patches_all.append(content_patches)

t1_patches_all = np.array(t1_patches_all)  # (n_subjects, n_patches, n_content)
t2_patches_all = np.array(t2_patches_all)

n_subjects, n_patches, n_content = t1_patches_all.shape
print(f"\nExtracted: {n_subjects} subjects x {n_patches} patches x {n_content} content dims")

# ── Build gallery and queries ────────────────────────────────────────────────
# Query: every T1 patch  (n_subjects * n_patches, n_content)
# Gallery: every T2 patch (n_subjects * n_patches, n_content)
queries = t1_patches_all.reshape(-1, n_content)  # (N*P, C)
gallery = t2_patches_all.reshape(-1, n_content)  # (N*P, C)

# L2-normalise for cosine similarity
q_norm = queries / (np.linalg.norm(queries, axis=1, keepdims=True) + 1e-8)
g_norm = gallery / (np.linalg.norm(gallery, axis=1, keepdims=True) + 1e-8)

# Cosine similarity matrix: (N*P, N*P)
sim_matrix = q_norm @ g_norm.T

# Ground-truth: query i should match gallery i (same subject, same position)
# Subject ID for each patch
query_subject = np.repeat(np.arange(n_subjects), n_patches)
gallery_subject = np.repeat(np.arange(n_subjects), n_patches)
# Position ID for each patch
query_position = np.tile(np.arange(n_patches), n_subjects)
gallery_position = np.tile(np.arange(n_patches), n_subjects)

# ── Recall@k computation ────────────────────────────────────────────────────
# Sort gallery indices by descending similarity for each query
ranked = np.argsort(-sim_matrix, axis=1)

results = {}
for k in TOP_K:
    top_k_indices = ranked[:, :k]

    # Position-accurate: correct subject AND correct position
    top_k_subj = gallery_subject[top_k_indices]
    top_k_pos = gallery_position[top_k_indices]
    pos_correct = ((top_k_subj == query_subject[:, None]) & (top_k_pos == query_position[:, None])).any(axis=1)
    pos_recall = pos_correct.mean()

    # Subject-accurate: correct subject, any position
    subj_correct = (top_k_subj == query_subject[:, None]).any(axis=1)
    subj_recall = subj_correct.mean()

    results[k] = {"position": pos_recall, "subject": subj_recall}

# Chance levels
chance_pos = 1.0 / (n_subjects * n_patches)
chance_subj = n_patches / (n_subjects * n_patches)

# ── Print results ────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Cross-Modal Patch Retrieval (T1 query -> T2 gallery)")
print(f"{'='*60}")
print(f"  Gallery size: {n_subjects} subjects x {n_patches} patches = {n_subjects * n_patches}")
print(f"  Chance (position): {chance_pos:.4f}")
print(f"  Chance (subject):  {chance_subj:.4f}")
print()
for k in TOP_K:
    r = results[k]
    print(f"  Recall@{k:>2d}  position: {r['position']:.4f}   subject: {r['subject']:.4f}")

# ── Per-subject retrieval accuracy breakdown ─────────────────────────────────
print(f"\nPer-subject position Recall@1:")
for s in range(min(n_subjects, 20)):  # show at most 20
    mask = query_subject == s
    s_correct = (gallery_subject[ranked[mask, 0]] == s) & (gallery_position[ranked[mask, 0]] == query_position[mask])
    label = label_names[items[s]["label"]]
    print(f"  Subject {s:3d} ({label:>4s}): {s_correct.mean():.3f}")

# ── Heatmap: mean position-wise retrieval similarity ─────────────────────────
# For each spatial position, average the same-subject cosine similarity
pos_sim = np.zeros(n_patches)
for p in range(n_patches):
    sims = []
    for s in range(n_subjects):
        qi = s * n_patches + p
        gi = s * n_patches + p
        sims.append(sim_matrix[qi, gi])
    pos_sim[p] = np.mean(sims)

gD, gH, gW = PATCH_GRID
pos_sim_vol = pos_sim.reshape(gD, gH, gW)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    "Mean Same-Subject Cosine Similarity by Spatial Position\n" "(T1 content vs T2 content, per patch)",
    fontsize=13,
    fontweight="bold",
)

mid_d, mid_h, mid_w = gD // 2, gH // 2, gW // 2
for ax, (title, slc) in zip(
    axes,
    [
        (f"Axial (d={mid_d})", pos_sim_vol[mid_d]),
        (f"Coronal (h={mid_h})", pos_sim_vol[:, mid_h]),
        (f"Sagittal (w={mid_w})", pos_sim_vol[:, :, mid_w]),
    ],
):
    im = ax.imshow(slc, cmap="RdYlGn", vmin=0, vmax=1, interpolation="nearest", aspect="auto")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print(f"\nSpatial similarity range: {pos_sim.min():.3f} — {pos_sim.max():.3f} (mean {pos_sim.mean():.3f})")

## 7b-iii. PCA / t-SNE of Spatial Patches — Colored by Position & Modality

Visualise whether patches from the **same spatial position** across modalities cluster together.

- **Left**: colored by **spatial region** (coarse grid position) — same-position patches from T1 and T2
  should overlap if content is aligned
- **Right**: colored by **modality** (T1 vs T2) — if content is aligned, there should be NO
  visible separation by modality; patches should intermix

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ── Re-use t1_patches_all / t2_patches_all from previous cell ───────────────
# Shape: (n_subjects, n_patches, n_content)
# Subsample subjects if dataset is large (t-SNE is O(n^2))
MAX_SUBJECTS = 50
if n_subjects > MAX_SUBJECTS:
    rng = np.random.RandomState(42)
    sub_idx = rng.choice(n_subjects, MAX_SUBJECTS, replace=False)
    t1_sub = t1_patches_all[sub_idx]
    t2_sub = t2_patches_all[sub_idx]
    n_sub = MAX_SUBJECTS
else:
    t1_sub = t1_patches_all
    t2_sub = t2_patches_all
    n_sub = n_subjects

# Stack T1 and T2 patches: (2 * n_sub * n_patches, n_content)
all_patches = np.concatenate([t1_sub.reshape(-1, n_content), t2_sub.reshape(-1, n_content)], axis=0)

# Labels
n_pts_per_mod = n_sub * n_patches
modality_labels = np.array(["T1"] * n_pts_per_mod + ["T2"] * n_pts_per_mod)
position_labels = np.tile(np.arange(n_patches), n_sub * 2)  # spatial position index
subject_labels = np.concatenate([np.repeat(np.arange(n_sub), n_patches)] * 2)

# ── PCA ──────────────────────────────────────────────────────────────────────
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(all_patches)

# ── t-SNE ────────────────────────────────────────────────────────────────────
print(f"Running t-SNE on {len(all_patches)} patches ...")
tsne = TSNE(
    n_components=2, perplexity=min(30, len(all_patches) // 4), random_state=42, init="pca", learning_rate="auto"
)
tsne_coords = tsne.fit_transform(all_patches)
print("Done.")

# ── Assign colors for spatial position (use a cyclic colormap) ───────────────
# Group positions into coarse regions for visual clarity
gD, gH, gW = PATCH_GRID
# Use depth slice as region label (most interpretable for brain volumes)
pos_depth = position_labels // (gH * gW)  # which depth slice
pos_colors = plt.cm.tab10(pos_depth / max(gD - 1, 1))

mod_colors = np.where(
    modality_labels[:, None] == "T1", [0.2, 0.5, 0.9, 0.5], [0.9, 0.3, 0.2, 0.5]  # blue, semi-transparent
)  # red, semi-transparent

# ── Plot PCA ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f"PCA of Spatial Content Patches ({n_sub} subjects, {n_patches} patches each)\n"
    f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, "
    f"PC2={pca.explained_variance_ratio_[1]:.1%}",
    fontsize=13,
    fontweight="bold",
)

ax = axes[0]
ax.scatter(pca_coords[:, 0], pca_coords[:, 1], c=pos_colors, s=8, alpha=0.5, edgecolors="none")
ax.set_title(f"Colored by depth slice (0..{gD-1})")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
# Legend for depth slices
for d in range(gD):
    ax.scatter([], [], c=[plt.cm.tab10(d / max(gD - 1, 1))], s=40, label=f"depth {d}")
ax.legend(fontsize=8, markerscale=1.5)

ax = axes[1]
t1_mask = modality_labels == "T1"
ax.scatter(pca_coords[t1_mask, 0], pca_coords[t1_mask, 1], c="steelblue", s=8, alpha=0.4, edgecolors="none", label="T1")
ax.scatter(pca_coords[~t1_mask, 0], pca_coords[~t1_mask, 1], c="tomato", s=8, alpha=0.4, edgecolors="none", label="T2")
ax.set_title("Colored by modality")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(fontsize=10, markerscale=3)

plt.tight_layout()
plt.show()

# ── Plot t-SNE ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f"t-SNE of Spatial Content Patches ({n_sub} subjects, {n_patches} patches each)", fontsize=13, fontweight="bold"
)

ax = axes[0]
ax.scatter(tsne_coords[:, 0], tsne_coords[:, 1], c=pos_colors, s=8, alpha=0.5, edgecolors="none")
ax.set_title(f"Colored by depth slice (0..{gD-1})")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
for d in range(gD):
    ax.scatter([], [], c=[plt.cm.tab10(d / max(gD - 1, 1))], s=40, label=f"depth {d}")
ax.legend(fontsize=8, markerscale=1.5)

ax = axes[1]
ax.scatter(
    tsne_coords[t1_mask, 0], tsne_coords[t1_mask, 1], c="steelblue", s=8, alpha=0.4, edgecolors="none", label="T1"
)
ax.scatter(
    tsne_coords[~t1_mask, 0], tsne_coords[~t1_mask, 1], c="tomato", s=8, alpha=0.4, edgecolors="none", label="T2"
)
ax.set_title("Colored by modality")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(fontsize=10, markerscale=3)

plt.tight_layout()
plt.show()

# ── Quantitative: silhouette score for modality separation ───────────────────
from sklearn.metrics import silhouette_score

# Subsample for speed if needed
MAX_SIL = 5000
if len(all_patches) > MAX_SIL:
    rng = np.random.RandomState(0)
    sil_idx = rng.choice(len(all_patches), MAX_SIL, replace=False)
else:
    sil_idx = np.arange(len(all_patches))

sil_mod = silhouette_score(all_patches[sil_idx], modality_labels[sil_idx], metric="cosine")
sil_pos = silhouette_score(all_patches[sil_idx], position_labels[sil_idx], metric="cosine")

print(f"\nSilhouette scores (cosine, higher = more separated):")
print(
    f"  By modality:  {sil_mod:+.3f}  {'(BAD: modalities still separated)' if sil_mod > 0.1 else '(GOOD: modalities intermixed)'}"
)
print(
    f"  By position:  {sil_pos:+.3f}  {'(GOOD: spatial structure preserved)' if sil_pos > 0.05 else '(WARNING: no spatial structure)'}"
)